In [ ]:
%%writefile layer1_cnn.py
import os
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

def get_demeter_augmenter(img_height, img_width):
    return models.Sequential([
        layers.Rescaling(1./127.5, offset=-1, input_shape=(img_height, img_width, 3)),
        layers.RandomRotation(0.2),
        layers.RandomFlip("horizontal"),
        layers.RandomZoom(0.1),
    ], name="demeter_augmenter_pipeline")

def preprocess_tfds(image, label, img_height, img_width):
    image = tf.image.resize(image, [img_height, img_width])
    return image, label

def train_and_save_cnn(save_path, dataset_type='tfds', kaggle_train_dir=None, kaggle_val_dir=None, img_height=224, img_width=224, epochs=5, batch_size=64):
    print(f"Initializing Layer 1: CNN Training Pipeline using {dataset_type}...")
    
    # -------------------------------------------
    # DATASET PIPELINE A: TensorFlow Plant Leaves
    # -------------------------------------------
    if dataset_type == 'tfds':
        (ds_train), ds_info = tfds.load('plant_leaves', split='train', as_supervised=True, with_info=True)
        class_names = ds_info.features['label'].names
        num_classes = ds_info.features['label'].num_classes
        
        dataset_size = ds_info.splits['train'].num_examples
        train_size = int(0.8 * dataset_size)
        
        ds_train = ds_train.shuffle(1000, seed=123)
        train_ds = ds_train.take(train_size).map(lambda x, y: preprocess_tfds(x, y, img_height, img_width), num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)
        val_ds = ds_train.skip(train_size).map(lambda x, y: preprocess_tfds(x, y, img_height, img_width), num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # -------------------------------------------
    # DATASET PIPELINE B: PlantNet-300K
    # -------------------------------------------
    elif dataset_type == 'kaggle':
        if not kaggle_train_dir or not kaggle_val_dir:
            raise ValueError("You must provide both train and val directories for PlantNet-300K.")
            
        print("Loading PlantNet-300K Native Splits...")
        # PlantNet-300K is huge, so we just load from the pre-existing train/val folders
        train_ds = tf.keras.utils.image_dataset_from_directory(
            kaggle_train_dir, seed=123,
            image_size=(img_height, img_width), batch_size=batch_size)
        
        val_ds = tf.keras.utils.image_dataset_from_directory(
            kaggle_val_dir, seed=123,
            image_size=(img_height, img_width), batch_size=batch_size)
            
        class_names = train_ds.class_names
        num_classes = len(class_names)
    
    else:
        raise ValueError("dataset_type must be 'tfds' or 'kaggle'")

    print(f"Successfully loaded {num_classes} classes.")

    # -------------------------------------------
    # ARCHITECTURE & TRAINING
    # -------------------------------------------
    base_model = MobileNetV2(input_shape=(img_height, img_width, 3), include_top=False, weights='imagenet')
    base_model.trainable = False 

    model = models.Sequential([
        get_demeter_augmenter(img_height, img_width),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    model.fit(train_ds, validation_data=val_ds, epochs=epochs)
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    model.save(save_path)
    print(f"CNN successfully saved to {save_path}")
    return class_names

In [ ]:
%%writefile layer3_rf.py
import os
import pandas as pd
import joblib 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def train_and_save_rf(csv_dataset_path, save_path):
    print("Initializing Layer 3: Random Forest Training Pipeline...")
    
    df = pd.read_csv(csv_dataset_path)
    
    # Feature selection based on your Kaggle dataset schema
    X = df[['Species_Code', 'Temp', 'Moisture', 'Light']] 
    y = df[['Needs_Water', 'Needs_Fertilizer', 'Needs_Light']] 
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    joblib.dump(rf_model, save_path)
    print(f"Random Forest successfully saved to {save_path}")

In [ ]:
from layer1_cnn import train_and_save_cnn
from layer3_rf import train_and_save_rf

CNN_SAVE_PATH = "/kaggle/working/models/layer1_cnn_model.keras"
RF_SAVE_PATH = "/kaggle/working/models/layer3_rf_model.pkl"

print("--- STARTING LAYER 1 (PlantNet-300K) ---")

# Exact Kaggle paths for the pre-split PlantNet-300K dataset
PLANTNET_TRAIN = "/kaggle/input/plantnet-300k-images/plantnet_300K/images_train" 
PLANTNET_VAL = "/kaggle/input/plantnet-300k-images/plantnet_300K/images_val" 

# Make sure to turn on a Kaggle GPU (T4 x2) in the notebook settings before running this!
classes = train_and_save_cnn(
    save_path=CNN_SAVE_PATH, 
    dataset_type='kaggle',
    kaggle_train_dir=PLANTNET_TRAIN,
    kaggle_val_dir=PLANTNET_VAL,
    epochs=10
)

print("\n--- STARTING LAYER 3 ---")
# KAGGLE_TABULAR_CSV = "/kaggle/input/plant-growth-data-classification/plant_growth_data.csv"
# train_and_save_rf(csv_dataset_path=KAGGLE_TABULAR_CSV, save_path=RF_SAVE_PATH)

print("\nAll models trained and ready for download!")